# Reproduce and reduce failures: practice

Reproduce → simplify to a small case → inspect the exception → distinguish causes → fix and repeat the same case.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [guides/08_debugging.ipynb](../../../../guides/08_debugging.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Reproduce, inspect, simplify, fix

A small failing case separates causes. Catch the exact exception in a teaching example so the notebook can continue, then implement a specific repair.

**Predict:** Will the original operation raise NameError or KeyError?

In [ ]:
record = {"title": "Agents"}
try:
    print(record["author"])
except KeyError as error:
    print("Reproduced:", type(error).__name__)
author = record.get("author", "Unknown")
print("Fixed optional lookup:", author)
assert author == "Unknown"

**Check:** KeyError because the dictionary exists but the key does not. NameError means the variable itself was not defined.

**Change one thing:** Remove the record definition in a scratch cell. Explain how the new NameError differs from the original missing-key error.

## 2. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 3. Request, status, and parsed body

An HTTP client sends a request and returns a status plus response body. Provider requests are network side effects; a fixture lets you practice handling success/failure without sending one.

**Predict:** Will a non-200 response be treated as success here?

In [ ]:
fixtures = [{"status": 200, "body": {"result": "fictional success"}},
            {"status": 401, "body": {"error": "authentication rejected"}}]
for response in fixtures:
    if response["status"] >= 400:
        print("Failure status:", response["status"])
    else:
        print("Parsed result:", response["body"]["result"])
assert fixtures[1]["status"] == 401

**Check:** Status values below 400 enter this fixture success branch; 401 is rejected. Use the provider response to distinguish causes.

**Change one thing:** Add a 429 fixture and distinguish rate limiting from authentication. Explain why real calls need timeouts.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.